In [159]:
!pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
	from google.colab import drive
	drive.mount('/content/drive')

	import os
	os.chdir('/content/drive/MyDrive/파트4')
	print('✅ Succesful access google_drive_directory')

except Exception as e:
	print('❌')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Succesful access google_drive_directory


In [160]:
API_KEY_PATH = '/content/drive/MyDrive/파트4/sprintda03-yujin.json'

def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name

In [161]:
## literal_eval 형변환 함수
def to_literal_eval(df, column):
    df[column] = df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])

## 삭제유저
drop_users = [831956, 1580627, 1580689, 1580626, 995177]

In [162]:
polls_questionset = get_df('votes','polls_questionset')

In [163]:
# 컬럼순서 정리
polls_questionset = polls_questionset[['id', 'user_id', 'question_piece_id_list', 'status', 'created_at', 'opening_time']]

# 리스트타입 변환
to_literal_eval(polls_questionset, 'question_piece_id_list')

# 셋타입으로 변환
polls_questionset['question_piece_id_list'] = polls_questionset['question_piece_id_list'].apply(lambda x: frozenset(x))

# 시간타입 변환
polls_questionset['opening_time'] = pd.to_datetime(polls_questionset['opening_time'])
polls_questionset['created_at'] = pd.to_datetime(polls_questionset['created_at'])

# 관리자 유저 삭제
drop_users = [831956, 1580627, 1580689, 1580626, 995177]
polls_questionset = polls_questionset[~polls_questionset['user_id'].isin(drop_users)]

In [164]:
# 파생컬럼
polls_questionset['time_diff'] = polls_questionset['opening_time'] - polls_questionset['created_at']
polls_questionset['time_diff_second'] = polls_questionset['time_diff'].apply(lambda x: x.total_seconds())

## info

In [165]:
polls_questionset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158384 entries, 0 to 158383
Data columns (total 8 columns):
 #   Column                  Non-Null Count   Dtype          
---  ------                  --------------   -----          
 0   id                      158384 non-null  int64          
 1   user_id                 158384 non-null  int64          
 2   question_piece_id_list  158384 non-null  object         
 3   status                  158384 non-null  object         
 4   created_at              158384 non-null  datetime64[ns] 
 5   opening_time            158384 non-null  datetime64[ns] 
 6   time_diff               158384 non-null  timedelta64[ns]
 7   time_diff_second        158384 non-null  float64        
dtypes: datetime64[ns](2), float64(1), int64(2), object(2), timedelta64[ns](1)
memory usage: 9.7+ MB


## 결측치

In [166]:
polls_questionset.isna().sum()

,0
id,0
user_id,0
question_piece_id_list,0
status,0
created_at,0
opening_time,0
time_diff,0
time_diff_second,0


## 중복치

In [167]:
polls_questionset.loc[:,'user_id':].duplicated().sum()

0

In [168]:
polls_questionset.loc[:,'user_id':'question_piece_id_list'].duplicated().sum()

0

In [208]:
len(polls_questionset)

158384

## user_id

In [204]:
polls_questionset['user_id'].value_counts().reset_index()

,user_id,count
0,952220,370
1,849103,286
2,1184703,215
3,1162477,203
4,1213990,202
...,...,...
4967,880568,1
4968,973021,1
4969,1533302,1
4970,935820,1


In [207]:
df = polls_questionset['user_id'].value_counts().reset_index()
fig = px.violin(df, x='count', box=True, points='all')
fig.show()

## question_piece_id_list 질문집합 형태보기

In [169]:
print(polls_questionset['question_piece_id_list'].apply(len).unique())
print(polls_questionset['question_piece_id_list'].apply(lambda x: len(set(x))).unique())
'''내부적으로 고유한 질문리스트로 10개씩 잘 이루어져있음'''

[10]
[10]


'내부적으로 고유한 질문리스트로 10개씩 잘 이루어져있음'

In [218]:
print('중복되는 질문셋존재하지 않음')
polls_questionset[polls_questionset['question_piece_id_list'].duplicated(keep=False)]

중복되는 질문셋존재하지 않음


,id,user_id,question_piece_id_list,status,created_at,opening_time,time_diff,time_diff_second,time_diff_sign


## status 닫기(C), 열기(O), 종료(F)

In [235]:
pd.merge(polls_questionset['status'].value_counts().reset_index(),
         (polls_questionset['status'].value_counts(normalize=True)*100).reset_index(),
         on='status')

,status,count,proportion
0,F,153411,96.860163
1,O,4407,2.782478
2,C,566,0.357359


In [230]:
df = polls_questionset['status'].value_counts().reset_index()
fig = go.Figure(data=[go.Pie(labels=df['status'], values=df['count'])])
fig.update_layout(width=600, height=400)
fig.show()

## created_at

In [244]:
polls_questionset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158384 entries, 0 to 158383
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype          
---  ------                  --------------   -----          
 0   id                      158384 non-null  object         
 1   user_id                 158384 non-null  int64          
 2   question_piece_id_list  158384 non-null  object         
 3   status                  158384 non-null  object         
 4   created_at              158384 non-null  datetime64[ns] 
 5   opening_time            158384 non-null  datetime64[ns] 
 6   time_diff               158384 non-null  timedelta64[ns]
 7   time_diff_second        158384 non-null  float64        
 8   time_diff_sign          158384 non-null  object         
dtypes: datetime64[ns](2), float64(1), int64(1), object(4), timedelta64[ns](1)
memory usage: 10.9+ MB


In [253]:
polls_questionset['created_Y_M'] = polls_questionset['created_at'].dt.to_period('M').astype('str')
polls_questionset['created_Y_M_D'] = polls_questionset['created_at'].dt.to_period('D').astype('str')
polls_questionset['hour'] = polls_questionset['created_at'].dt.hour
polls_questionset['weekday'] = polls_questionset['created_at'].dt.weekday
polls_questionset['is_weekend'] = polls_questionset['created_at'].dt.weekday >= 5
polls_questionset['time_of_day'] = pd.cut(polls_questionset['created_at'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

polls_questionset = polls_questionset.sort_values(by='created_at')

In [283]:
created_Y_M = polls_questionset['created_Y_M'].value_counts().reset_index().sort_values(by='created_Y_M')
fig = px.line(created_Y_M, x='created_Y_M', y='count', title='Created At Time Series', markers=True)
fig.update_xaxes(title_text='Created At')
fig.update_yaxes(title_text='Created count')
fig.update_layout(width=1200, height=400)
fig.show()


created_Y_M_D = polls_questionset['created_Y_M_D'].value_counts().reset_index().sort_values(by='created_Y_M_D')
# created_2023_M_D = created_Y_M_D[created_Y_M_D['created_Y_M_D'].str[:4] == '2023']

fig = px.line(created_Y_M_D, x='created_Y_M_D', y='count', title='Created At Time Series', markers=True)
fig.update_xaxes(title_text='Created At')
fig.update_yaxes(title_text='Created count')
fig.update_layout(width=1700, height=600)
fig.show()

## opening_time

In [301]:
polls_questionset

polls_questionset['O_y_m'] = polls_questionset['opening_time'].dt.to_period('M').astype('str')
polls_questionset['O_y_m_d'] = polls_questionset['opening_time'].dt.to_period('D').astype('str')
polls_questionset['O_hour'] = polls_questionset['opening_time'].dt.hour
polls_questionset['O_weekday'] = polls_questionset['opening_time'].dt.weekday
polls_questionset['O_is_weekend'] = polls_questionset['opening_time'].dt.weekday >= 5
polls_questionset['O_time_of_day'] = pd.cut(polls_questionset['opening_time'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

In [302]:
O_y_m = polls_questionset['O_y_m'].value_counts().reset_index().sort_values(by='O_y_m')
fig = px.line(O_y_m, x='O_y_m', y='count', title='Opening At Time Series', markers=True)
fig.update_xaxes(title_text='Opening At')
fig.update_yaxes(title_text='Opening count')
fig.update_layout(width=1500, height=400)
fig.show()


O_y_m_d = polls_questionset['O_y_m_d'].value_counts().reset_index().sort_values(by='O_y_m_d')
fig = px.line(O_y_m_d, x='O_y_m_d', y='count', title='Opening At Time Series', markers=True)
fig.update_xaxes(title_text='Opening At')
fig.update_yaxes(title_text='Opening count')
fig.update_layout(width=1500, height=600)
fig.show()

## time_diff_second

In [300]:
len(polls_questionset.query('time_diff_second < 0'))

679

In [264]:
polls_questionset['hour'].value_counts().reset_index().sort_values(by='hour')

,hour,count
18,0,4016
17,1,4889
15,2,5452
14,3,5949
13,4,6114
10,5,6473
9,6,7123
8,7,8039
6,8,9346
7,9,9326


In [254]:
# # Plotly 시계열 그래프


In [221]:
OPEN_question = polls_questionset.query('status == "O"')
FINISH_question = polls_questionset.query('status == "F"')
CLOSE_question = polls_questionset.query('status == "C"')

## problem1: ['time_diff_second'] < 0

In [170]:
problem1 = polls_questionset[polls_questionset['time_diff_second'] < 0]

In [171]:
problem1['status'].value_counts()

,count
status,
F,671
O,8


In [172]:
problem1['time_diff_second'].value_counts().reset_index().T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
time_diff_second,-1.0,-2.0,-3.0,-5.0,-7.0,-4.0,-14.0,-10.0,-8.0,-6.0,-9.0,-12.0,-17.0,-11.0,-29.0
count,628.0,16.0,6.0,6.0,3.0,3.0,3.0,3.0,3.0,2.0,2.0,1.0,1.0,1.0,1.0


## 닫기(C), 열기(O), 종료(F)에따라 음수가 나오는 경우가 생기는 것인지

In [183]:
polls_questionset['time_diff_sign'] = polls_questionset['time_diff_second'].map(lambda x: 'positive' if x > 0 else ('negatve' if x < 0 else 'zero'))

array(['F', 'C', 'O'], dtype=object)

In [197]:
problem1_Close['time_diff_sign'].value_counts().reset_index(name='count_in_CLOSE')

,time_diff_sign,count_in_CLOSE
0,positive,566


In [198]:
problem1_Open['time_diff_sign'].value_counts().reset_index(name='count_in_OPEN')

,time_diff_sign,count_in_OPEN
0,positive,4274
1,zero,125
2,negatve,8


In [200]:
problem1_Finish['time_diff_sign'].value_counts().reset_index(name='count_in_FINISH')

,time_diff_sign,count_in_FINISH
0,positive,147827
1,zero,4913
2,negatve,671


In [174]:
problem1_check1['status'].value_counts()

,count
status,
F,4913
O,125


In [175]:
problem1_check2 = polls_questionset[polls_questionset['time_diff_second'] > 0]

In [176]:
problem1_check2['status'].value_counts()

,count
status,
F,147827
O,4274
C,566


In [177]:
polls_questionset

,id,user_id,question_piece_id_list,status,created_at,opening_time,time_diff,time_diff_second
0,99817,849436,"(998464, 998465, 998466, 998467, 998458, 998459, 998460, 998461, 998462, 998463)",F,2023-04-28 12:27:23,2023-04-28 12:27:22,-1 days +23:59:59,-1.0
1,99830,849438,"(998592, 998593, 998594, 998595, 998596, 998597, 998588, 998589, 998590, 998591)",F,2023-04-28 12:28:07,2023-04-28 12:28:07,0 days 00:00:00,0.0
2,99840,847375,"(998689, 998691, 998693, 998695, 998697, 998699, 998700, 998702, 998704, 998706)",F,2023-04-28 12:28:38,2023-04-28 12:28:38,0 days 00:00:00,0.0
3,99841,849446,"(998688, 998690, 998692, 998694, 998696, 998698, 998701, 998703, 998705, 998707)",F,2023-04-28 12:28:38,2023-04-28 12:28:38,0 days 00:00:00,0.0
4,99848,849477,"(998768, 998769, 998770, 998771, 998772, 998773, 998774, 998775, 998776, 998777)",F,2023-04-28 12:28:57,2023-04-28 12:28:57,0 days 00:00:00,0.0
...,...,...,...,...,...,...,...,...
158379,20838253,1251933,"(208383296, 208383297, 208383298, 208383299, 208383300, 208383291, 208383292, 208383293, 208383294, 208383295)",C,2024-05-05 14:06:27,2024-05-05 14:46:27,0 days 00:40:00,2400.0
158380,20838344,876072,"(208384201, 208384202, 208384203, 208384204, 208384205, 208384206, 208384207, 208384208, 208384209, 208384210)",C,2024-05-06 10:58:20,2024-05-06 11:38:20,0 days 00:40:00,2400.0
158381,20838419,1208878,"(208384960, 208384951, 208384952, 208384953, 208384954, 208384955, 208384956, 208384957, 208384958, 208384959)",C,2024-05-07 00:15:00,2024-05-07 00:55:00,0 days 00:40:00,2400.0
158382,20838445,1001607,"(208385216, 208385217, 208385218, 208385219, 208385220, 208385211, 208385212, 208385213, 208385214, 208385215)",C,2024-05-07 11:29:08,2024-05-07 12:09:08,0 days 00:40:00,2400.0
